nrms

In [ ]:
'''
run this to clear ouput

import os
import shutil

folders_to_remove = [
    CFG["cache_dir"],
    "/kaggle/working/output",
]

for folder in folders_to_remove:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print("Removed folder:", folder)
    else:
        print("Not found:", folder)

# tạo lại folder trống nếu cần
os.makedirs(CFG["cache_dir"], exist_ok=True)
os.makedirs("/kaggle/working/output", exist_ok=True)

print("Recreated empty folders.")'''

NOTE:
1. distil bert
2. finetuned bert with mind small
3. uses word attention instead of cls
4. for each article, uses attention to get our represent vector from (abstract, title, entity)

cell 2 import

In [2]:
import os, math, json, random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel

CELL 3 — Config

In [ ]:
CFG = {
    # BERT backbone
    "bert_name": "distilbert-base-uncased",
    "max_title_len": 32,
    "max_abs_len": 64,

    # NRMS heads (giữ style NRMS)
    "num_head_text": 16,
    "num_head_entity": 8,
    "text_attn_vector_size": 200,
    "entity_attn_vector_size": 100,
    "news_final_attn_vector_size": 200,
    "news_final_embed_size": 400,
    "his_final_attn_vector_size": 200,
    "user_embed_size": 400,

    # Training
    "neg_sample_num": 4,
    "batch_size": 64,
    "lr": 1e-4,
    "epochs": 6,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # Cache
    "cache_dir": "./cache_nrms_bert",
    "use_fp16_cache": True,   # cache token embeddings float16 để nhẹ
}
os.makedirs(CFG["cache_dir"], exist_ok=True)
CFG

{'bert_name': '/kaggle/input/notebooks/xuntngtrn/finetune-bert-with-mind-small/finetuned_distilbert_nrms',
 'max_title_len': 32,
 'max_abs_len': 64,
 'num_head_text': 16,
 'num_head_entity': 8,
 'text_attn_vector_size': 200,
 'entity_attn_vector_size': 100,
 'news_final_attn_vector_size': 200,
 'news_final_embed_size': 400,
 'his_final_attn_vector_size': 200,
 'user_embed_size': 400,
 'neg_sample_num': 4,
 'batch_size': 64,
 'lr': 0.0001,
 'epochs': 6,
 'device': 'cuda',
 'cache_dir': './cache_nrms_bert',
 'use_fp16_cache': True}

save cache

In [8]:
CFG["cache_dir"] = "/kaggle/working/cache_nrms_bert"
os.makedirs(CFG["cache_dir"], exist_ok=True)

TITLE_CACHE = os.path.join(CFG["cache_dir"], f"title_{CFG['bert_name'].replace('/','_')}_{CFG['max_title_len']}.pt")
ABS_CACHE   = os.path.join(CFG["cache_dir"], f"abstract_{CFG['bert_name'].replace('/','_')}_{CFG['max_abs_len']}.pt")

print("TITLE_CACHE:", TITLE_CACHE)
print("ABS_CACHE:", ABS_CACHE)

TITLE_CACHE: /kaggle/working/cache_nrms_bert/title__kaggle_input_notebooks_xuntngtrn_finetune-bert-with-mind-small_finetuned_distilbert_nrms_32.pt
ABS_CACHE: /kaggle/working/cache_nrms_bert/abstract__kaggle_input_notebooks_xuntngtrn_finetune-bert-with-mind-small_finetuned_distilbert_nrms_64.pt


CELL 4 — Paths

In [7]:
TRAIN_DIR = '/kaggle/input/datasets/xuntngtrn/mind-large-train'
DEV_DIR   = '/kaggle/input/datasets/xuntngtrn/ming-large-dev'

TRAIN_ENTITY_VEC = os.path.join(TRAIN_DIR, 'entity_embedding.vec')
DEV_ENTITY_VEC   = os.path.join(DEV_DIR, 'entity_embedding.vec')

CELL 5 — Load news & behaviors

In [8]:
def load_news(path_dir):
    cols = ['NewsID','Category','SubCategory','Title','Abstract','URL','TitleEntities','AbstractEntities']
    df = pd.read_csv(os.path.join(path_dir,'news.tsv'), sep='\t', header=None)
    df.columns = cols
    df['Title'] = df['Title'].fillna("")
    df['Abstract'] = df['Abstract'].fillna("")
    df['TitleEntities'] = df['TitleEntities'].fillna("[]")
    df['AbstractEntities'] = df['AbstractEntities'].fillna("[]")
    return df

def load_behaviors(path_dir):
    cols = ['ImpressionID','UserID','Time','History','Impressions']
    df = pd.read_csv(os.path.join(path_dir,'behaviors.tsv'), sep='\t', header=None)
    df.columns = cols
    df['History'] = df['History'].fillna("")
    return df



In [9]:
news_train = load_news(TRAIN_DIR)
news_dev   = load_news(DEV_DIR)
beh_train  = load_behaviors(TRAIN_DIR)
beh_dev    = load_behaviors(DEV_DIR)

print("news_train:", news_train.shape, "| beh_train:", beh_train.shape)
print("news_dev:", news_dev.shape, "| beh_dev:", beh_dev.shape)
news_train.head(2)

news_train: (101527, 8) | beh_train: (2232748, 5)
news_dev: (72023, 8) | beh_dev: (376471, 5)


,NewsID,Category,SubCategory,Title,Abstract,URL,TitleEntities,AbstractEntities
0,N88753,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N45436,news,newsscienceandtechnology,Walmart Slashes Prices on Last-Generation iPads,Apple's new iPad releases bring big deals on l...,https://assets.msn.com/labs/mind/AABmf2I.html,"[{""Label"": ""IPad"", ""Type"": ""J"", ""WikidataId"": ...","[{""Label"": ""IPad"", ""Type"": ""J"", ""WikidataId"": ..."


CELL 6 — Category/SubCategory mapping

In [11]:
def build_cat_maps(news_df):
    cats = sorted(set(news_df['Category'].astype(str)))
    subcats = sorted(set(news_df['SubCategory'].astype(str)))
    cat2id = {c:i+1 for i,c in enumerate(cats)}       # 0 = PAD
    subcat2id = {c:i+1 for i,c in enumerate(subcats)} # 0 = PAD
    return cat2id, subcat2id



In [12]:
cat2id, subcat2id = build_cat_maps(pd.concat([news_train, news_dev], ignore_index=True))
num_cat, num_subcat = len(cat2id), len(subcat2id)
num_cat, num_subcat

(18, 285)

CELL 7 — Entity embedding (giữ như NRMS gốc)

In [13]:
def load_entity_embedding(vec_paths):
    for p in vec_paths:
        if os.path.exists(p):
            chosen = p
            break
    else:
        raise FileNotFoundError("Không tìm thấy entity_embedding.vec ở TRAIN/DEV.")

    vectors = []
    with open(chosen, 'r', encoding='utf-8') as f:
        first_line = f.readline().strip().split()

        # Nếu dòng đầu là header: "N dim"
        if len(first_line) == 2 and all(tok.isdigit() for tok in first_line):
            _, dim = map(int, first_line)
        else:
            # Không phải header -> đọc lại từ đầu
            f.seek(0)
            dim = None

        for line in f:
            arr = line.strip().split()
            if not arr:
                continue

            # Case A: dòng là "Q41 0.12 0.03 ...": bỏ token đầu
            # Case B: dòng là "0.12 0.03 ..." : giữ nguyên
            start_idx = 0
            try:
                float(arr[0])
            except:
                start_idx = 1

            vec = list(map(float, arr[start_idx:]))

            if dim is None:
                dim = len(vec)
            # nếu có dòng bị lệch dim thì skip cho an toàn
            if len(vec) != dim:
                continue

            vectors.append(vec)

    # 0 = PAD entity
    embed = np.zeros((len(vectors) + 1, dim), dtype=np.float32)
    embed[1:] = np.array(vectors, dtype=np.float32)
    return embed

In [14]:
entity_embed = load_entity_embedding([TRAIN_ENTITY_VEC, DEV_ENTITY_VEC])
print("entity_embed OK:", entity_embed.shape)

entity_embed OK: (42008, 100)


CELL 8 — Parse entities (tối giản)

In [15]:
def parse_entities(json_str, max_entity=10):
    """
    Tối giản: nếu entities json đã có EntityId/index thì lấy.
    Nếu không có -> trả 0 (PAD).
    """
    try:
        arr = json.loads(json_str)
    except Exception:
        arr = []
    ids = []
    for e in arr:
        if isinstance(e, dict):
            for k in ["EntityId", "entity_id", "entityId", "Index", "index"]:
                if k in e and isinstance(e[k], (int, float)):
                    ids.append(int(e[k]))
                    break
    ids = ids[:max_entity]
    if len(ids) < max_entity:
        ids += [0]*(max_entity-len(ids))
    return np.array(ids, dtype=np.int64).reshape(max_entity, 1)

MAX_ENTITY = 10

CELL 9 — BERT cache token embeddings cho Title & Abstract

In [16]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert = AutoModel.from_pretrained(CFG["bert_name"]).to(CFG["device"])
bert_hidden = bert.config.hidden_size
bert_hidden

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: /kaggle/input/notebooks/xuntngtrn/finetune-bert-with-mind-small/finetuned_distilbert_nrms
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


768

In [40]:
@torch.no_grad()
def build_bert_cache(news_df, cache_path, text_col, max_len, batch_size=64):
    news_ids = news_df["NewsID"].astype(str).tolist()
    texts = news_df[text_col].astype(str).tolist()
    all_embs, all_masks = [], []
    
    bert.eval()
    for i in tqdm(range(0, len(texts), batch_size), desc=f"BERT cache {text_col}"):
        batch_text = texts[i:i+batch_size]
        enc = tokenizer(
            batch_text,
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].to(CFG["device"])
        attn_mask = enc["attention_mask"].to(CFG["device"])
        
        out = bert(input_ids=input_ids, attention_mask=attn_mask)
        tok = out.last_hidden_state  # (B,L,H)
        tok = tok.half().cpu() if CFG["use_fp16_cache"] else tok.float().cpu()
        all_embs.append(tok)
        all_masks.append(attn_mask.cpu().to(torch.float32))
        
    token_embs = torch.cat(all_embs, dim=0)
    attn_masks = torch.cat(all_masks, dim=0)
    obj = {"news_ids": news_ids, "token_embs": token_embs, "attn_masks": attn_masks}
    torch.save(obj, cache_path)
    return obj



In [ ]:
news_all = pd.concat([news_train, news_dev], ignore_index=True).drop_duplicates("NewsID")
print("Total unique news:", len(news_all))

up cache manually

In [17]:
TITLE_CACHE = '/kaggle/input/models/giahnhong/mind-title-cache/tensorflow2/default/1/title_32.pt'
ABS_CACHE   = '/kaggle/input/models/giahnhong/mind-l-abs-cache/tensorflow2/default/1/abstract_64.pt'

title_cache = torch.load(TITLE_CACHE, map_location="cpu")
abs_cache = torch.load(ABS_CACHE, map_location="cpu")
print(title_cache["token_embs"].shape, abs_cache["token_embs"].shape)

torch.Size([104151, 32, 768]) torch.Size([104151, 64, 768])


buil cache

In [ ]:
'''TITLE_CACHE = os.path.join(CFG["cache_dir"], f"title_{CFG['bert_name'].replace('/','_')}_{CFG['max_title_len']}.pt")
ABS_CACHE   = os.path.join(CFG["cache_dir"], f"abstract_{CFG['bert_name'].replace('/','_')}_{CFG['max_abs_len']}.pt")

if not os.path.exists(TITLE_CACHE):
    title_cache = build_bert_cache(news_all, TITLE_CACHE, "Title", CFG["max_title_len"], batch_size=64)
else:
    title_cache = torch.load(TITLE_CACHE, map_location="cpu")

if not os.path.exists(ABS_CACHE):
    abs_cache = build_bert_cache(news_all, ABS_CACHE, "Abstract", CFG["max_abs_len"], batch_size=32)
else:
    abs_cache = torch.load(ABS_CACHE, map_location="cpu")

print(title_cache["token_embs"].shape, abs_cache["token_embs"].shape)'''

In [17]:
import os

def safe_size(p):
    return os.path.getsize(p) if os.path.exists(p) else None

print("TITLE exists/size:", os.path.exists(TITLE_CACHE), safe_size(TITLE_CACHE))
print("ABS   exists/size:", os.path.exists(ABS_CACHE), safe_size(ABS_CACHE))
print("TITLE path:", TITLE_CACHE)
print("ABS path:", ABS_CACHE)

NameError: name 'TITLE_CACHE' is not defined

In [19]:
nid2idx = {nid:i for i,nid in enumerate(title_cache["news_ids"])}

def get_bert_by_nid(nid: str, which: str):
    # PAD/unknown
    if nid is None or nid=="" or nid=="PAD" or nid not in nid2idx:
        L = CFG["max_title_len"] if which=="title" else CFG["max_abs_len"]
        H = bert_hidden
        tok = torch.zeros((L,H), dtype=torch.float16 if CFG["use_fp16_cache"] else torch.float32)
        msk = torch.zeros((L,), dtype=torch.float32)
        return tok, msk
    
    i = nid2idx[nid]
    if which=="title":
        return title_cache["token_embs"][i], title_cache["attn_masks"][i]
    return abs_cache["token_embs"][i], abs_cache["attn_masks"][i]

CELL 10 — Negative sampling (giữ NRMS)

In [24]:
news_lookup = {row["NewsID"]: row for _,row in news_all.iterrows()}

In [20]:


def build_news_feat(nid: str):
    if nid=="PAD" or nid not in news_lookup:
        return {
            "cat": 0, "subcat": 0,
            "title_tok": get_bert_by_nid("PAD","title")[0],
            "title_mask": get_bert_by_nid("PAD","title")[1],
            "abs_tok": get_bert_by_nid("PAD","abs")[0],
            "abs_mask": get_bert_by_nid("PAD","abs")[1],
            "title_ent": torch.zeros((MAX_ENTITY,1), dtype=torch.long),
            "title_ent_mask": torch.zeros((MAX_ENTITY,), dtype=torch.float32),
            "abs_ent": torch.zeros((MAX_ENTITY,1), dtype=torch.long),
            "abs_ent_mask": torch.zeros((MAX_ENTITY,), dtype=torch.float32),
        }
    
    r = news_lookup[nid]
    cat = cat2id.get(str(r["Category"]), 0)
    subcat = subcat2id.get(str(r["SubCategory"]), 0)
    title_tok, title_mask = get_bert_by_nid(nid,"title")
    abs_tok, abs_mask = get_bert_by_nid(nid,"abs")
    
    title_ent = torch.from_numpy(parse_entities(r["TitleEntities"], MAX_ENTITY))
    abs_ent   = torch.from_numpy(parse_entities(r["AbstractEntities"], MAX_ENTITY))
    title_ent_mask = (title_ent[:,0]!=0).float()
    abs_ent_mask   = (abs_ent[:,0]!=0).float()
    
    return {
        "cat": cat, "subcat": subcat,
        "title_tok": title_tok, "title_mask": title_mask.float(),
        "abs_tok": abs_tok, "abs_mask": abs_mask.float(),
        "title_ent": title_ent.long(), "title_ent_mask": title_ent_mask,
        "abs_ent": abs_ent.long(), "abs_ent_mask": abs_ent_mask,
    }



In [25]:
news_feat = {}
for nid in tqdm(title_cache["news_ids"], desc="Build news_feat"):
    news_feat[nid] = build_news_feat(nid)
news_feat["PAD"] = build_news_feat("PAD")

news_feat["PAD"]["title_tok"].shape

Build news_feat: 100%|██████████| 104151/104151 [00:09<00:00, 10593.63it/s]


torch.Size([32, 768])

In [30]:
def preprocess_behaviors(behaviors: pd.DataFrame, neg_sample_num=4):
    samples = []
    for _, row in tqdm(behaviors.iterrows(), total=len(behaviors), desc="Preprocess behaviors"):
        imps = str(row["Impressions"]).strip().split()
        pos = [x.split("-")[0] for x in imps if x.endswith("-1")]
        neg = [x.split("-")[0] for x in imps if x.endswith("-0")]
        if not pos:
            continue
        
        history = str(row["History"]).strip().split()
        if len(history)==1 and history[0]=="":
            history = []
        
        for p in pos:
            sampled_neg = random.sample(neg, neg_sample_num) if len(neg) >= neg_sample_num else (neg + ["PAD"]*(neg_sample_num-len(neg)))
            cand = [p] + sampled_neg
            label = 0  # positive ở vị trí 0
            samples.append((history, cand, label))
    return samples



In [28]:
train_samples = preprocess_behaviors(beh_train, CFG["neg_sample_num"])
dev_samples   = preprocess_behaviors(beh_dev, CFG["neg_sample_num"])
len(train_samples), len(dev_samples)

NameError: name 'preprocess_behaviors' is not defined

CELL 11 — Build news_feat dict (cat/subcat + BERT title/abstract + entities)

CELL 12 — Dataset & DataLoader

In [26]:
MAX_HIS = 50

def pad_history(hist, max_his=MAX_HIS):
    if len(hist) >= max_his:
        return hist[-max_his:]
    return ["PAD"]*(max_his-len(hist)) + hist

class MindNrmsBertDataset(Dataset):
    def __init__(self, samples, news_feat):
        self.samples = samples
        self.news_feat = news_feat
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        history, cand, label = self.samples[idx]
        history = pad_history(history)
        
        h = [self.news_feat.get(nid, self.news_feat["PAD"]) for nid in history]
        c = [self.news_feat.get(nid, self.news_feat["PAD"]) for nid in cand]
        
        def stack(feats, key):
            return torch.stack([f[key] for f in feats], dim=0)
        
        return {
            # history
            "h_cat": torch.tensor([f["cat"] for f in h], dtype=torch.long),
            "h_subcat": torch.tensor([f["subcat"] for f in h], dtype=torch.long),
            "h_title_tok": stack(h, "title_tok"),
            "h_title_mask": stack(h, "title_mask"),
            "h_abs_tok": stack(h, "abs_tok"),
            "h_abs_mask": stack(h, "abs_mask"),
            "h_title_ent": stack(h, "title_ent"),
            "h_title_ent_mask": stack(h, "title_ent_mask"),
            "h_abs_ent": stack(h, "abs_ent"),
            "h_abs_ent_mask": stack(h, "abs_ent_mask"),
            
            # candidates (C=1+K)
            "c_cat": torch.tensor([f["cat"] for f in c], dtype=torch.long),
            "c_subcat": torch.tensor([f["subcat"] for f in c], dtype=torch.long),
            "c_title_tok": stack(c, "title_tok"),
            "c_title_mask": stack(c, "title_mask"),
            "c_abs_tok": stack(c, "abs_tok"),
            "c_abs_mask": stack(c, "abs_mask"),
            "c_title_ent": stack(c, "title_ent"),
            "c_title_ent_mask": stack(c, "title_ent_mask"),
            "c_abs_ent": stack(c, "abs_ent"),
            "c_abs_ent_mask": stack(c, "abs_ent_mask"),
            
            "label": torch.tensor(label, dtype=torch.long),
        }



In [27]:
train_ds = MindNrmsBertDataset(train_samples, news_feat)
dev_ds   = MindNrmsBertDataset(dev_samples, news_feat)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
dev_loader   = DataLoader(dev_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)

next(iter(train_loader))["h_title_tok"].shape

NameError: name 'train_samples' is not defined

CELL 13 — NRMS modules (giữ kiến trúc, thay GloVe bằng BERT token embs)

In [29]:
class MultiHeadedAttention(nn.Module):
    def __init__(self, head_num, model_dim, dropout=0.0):
        super().__init__()
        self.head_num = head_num
        self.model_dim = model_dim
        self.dim_per_head = model_dim // head_num
        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        # (B,L,D)
        B, L, D = query.size()
        q = query.view(B, L, self.head_num, self.dim_per_head).transpose(1, 2)
        k = key.view(B, L, self.head_num, self.dim_per_head).transpose(1, 2)
        v = value.view(B, L, self.head_num, self.dim_per_head).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.dim_per_head)
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1) == 0, -1e9)
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        x = torch.matmul(attn, v)
        x = x.transpose(1, 2).contiguous().view(B, L, D)
        return x


class TextEncoder(nn.Module):
    def __init__(self, word_embed_size, num_head, attn_vector_size):
        super().__init__()
        self.self_attn_size = word_embed_size // num_head * num_head
        
        self.weight_query = nn.Parameter(torch.FloatTensor(word_embed_size, self.self_attn_size))
        self.weight_key   = nn.Parameter(torch.FloatTensor(word_embed_size, self.self_attn_size))
        self.weight_value = nn.Parameter(torch.FloatTensor(word_embed_size, self.self_attn_size))
        self.trans_weight_v = nn.Parameter(torch.FloatTensor(self.self_attn_size, attn_vector_size))
        self.trans_weight_q = nn.Parameter(torch.FloatTensor(attn_vector_size, 1))
        
        self.self_attn = MultiHeadedAttention(num_head, word_embed_size)
        self._init_weights()

    def _init_weights(self):
        for p in [self.weight_query, self.weight_key, self.weight_value, self.trans_weight_v, self.trans_weight_q]:
            nn.init.uniform_(p, -0.1, 0.1)

    def forward(self, embeddings, mask_selfattn, mask_attn):
        q = torch.matmul(embeddings, self.weight_query)
        k = torch.matmul(embeddings, self.weight_key)
        v = torch.matmul(embeddings, self.weight_value)
        
        x = self.self_attn(q, k, v, mask_selfattn)
        scores = torch.matmul(torch.tanh(torch.matmul(x, self.trans_weight_v)), self.trans_weight_q).squeeze(-1)
        scores = scores.masked_fill(mask_attn == 0, -1e9)
        attend = F.softmax(scores, dim=1)
        attend = attend.masked_fill(mask_attn.sum(dim=1, keepdim=True) == 0, 0)
        embed = torch.bmm(x.transpose(1, 2), attend.unsqueeze(-1)).squeeze(-1)
        return embed

In [30]:
class NewsEncoder(nn.Module):
    def __init__(self, num_cat, num_subcat, entity_embed, cfg, bert_hidden):
        super().__init__()
        self.final_embed_size = cfg['news_final_embed_size']
        
        # Category/SubCategory embeddings (giữ NRMS)
        self.cat_embed = nn.Embedding(num_cat + 1, self.final_embed_size)
        self.subcat_embed = nn.Embedding(num_subcat + 1, self.final_embed_size)
        
        # Entity embedding (giữ NRMS)
        self.entity_embed = nn.Embedding.from_pretrained(torch.FloatTensor(entity_embed), freeze=True)
        
        # BERT token embeddings -> project dim để khớp số head
        self.title_in = bert_hidden
        self.abs_in   = bert_hidden
        self.title_proj_dim = (self.title_in // cfg['num_head_text']) * cfg['num_head_text']
        self.abs_proj_dim   = (self.abs_in   // cfg['num_head_text']) * cfg['num_head_text']
        
        self.title_proj = nn.Linear(self.title_in, self.title_proj_dim)
        self.abs_proj   = nn.Linear(self.abs_in,   self.abs_proj_dim)
        
        # Text encoders (giữ NRMS)
        self.title_encoder    = TextEncoder(self.title_proj_dim, cfg['num_head_text'], cfg['text_attn_vector_size'])
        self.abstract_encoder = TextEncoder(self.abs_proj_dim,   cfg['num_head_text'], cfg['text_attn_vector_size'])
        
        # Entity encoders (giữ NRMS)
        entity_size = entity_embed.shape[1] // cfg['num_head_entity'] * cfg['num_head_entity']
        self.title_entity_encoder    = TextEncoder(entity_embed.shape[1], cfg['num_head_entity'], cfg['entity_attn_vector_size'])
        self.abstract_entity_encoder = TextEncoder(entity_embed.shape[1], cfg['num_head_entity'], cfg['entity_attn_vector_size'])
        
        # Linear về final size (giữ NRMS)
        self.title_linear          = nn.Linear(self.title_proj_dim, self.final_embed_size)
        self.abstract_linear       = nn.Linear(self.abs_proj_dim,   self.final_embed_size)
        self.title_entity_linear   = nn.Linear(entity_size,         self.final_embed_size)
        self.abstract_entity_linear= nn.Linear(entity_size,         self.final_embed_size)
        
        # News-level additive attention (giữ NRMS)
        self.trans_weight_v = nn.Parameter(torch.FloatTensor(self.final_embed_size, cfg['news_final_attn_vector_size']))
        self.trans_weight_q = nn.Parameter(torch.FloatTensor(cfg['news_final_attn_vector_size'], 1))
        nn.init.uniform_(self.trans_weight_v, -0.1, 0.1)
        nn.init.uniform_(self.trans_weight_q, -0.1, 0.1)

    def forward(self, cat, subcat,
                title_tok, title_mask,
                abstract_tok, abstract_mask,
                title_ent, title_ent_mask,
                abstract_ent, abstract_ent_mask):

        cat_emb = self.cat_embed(cat).unsqueeze(1)
        subcat_emb = self.subcat_embed(subcat).unsqueeze(1)

        # Title (BERT token embs -> NRMS token-level attention)
        title_x = self.title_proj(title_tok.float())
        title_selfmask = title_mask.unsqueeze(1) * title_mask.unsqueeze(2)
        title_attn = self.title_encoder(title_x, title_selfmask, title_mask)
        title_attn = self.title_linear(title_attn).unsqueeze(1)

        # Abstract
        abs_x = self.abs_proj(abstract_tok.float())
        abs_selfmask = abstract_mask.unsqueeze(1) * abstract_mask.unsqueeze(2)
        abs_attn = self.abstract_encoder(abs_x, abs_selfmask, abstract_mask)
        abs_attn = self.abstract_linear(abs_attn).unsqueeze(1)

        # Entities
        title_ent_emb = self.entity_embed(title_ent[:, :, 0].long())
        title_ent_attn = self.title_entity_encoder(
            title_ent_emb,
            title_ent_mask.unsqueeze(1)*title_ent_mask.unsqueeze(2),
            title_ent_mask
        )
        title_ent_attn = self.title_entity_linear(title_ent_attn).unsqueeze(1)

        abs_ent_emb = self.entity_embed(abstract_ent[:, :, 0].long())
        abs_ent_attn = self.abstract_entity_encoder(
            abs_ent_emb,
            abstract_ent_mask.unsqueeze(1)*abstract_ent_mask.unsqueeze(2),
            abstract_ent_mask
        )
        abs_ent_attn = self.abstract_entity_linear(abs_ent_attn).unsqueeze(1)

        # News-level attention over [cat, subcat, title, abstract, title_ent, abs_ent]
        concat = torch.cat([cat_emb, subcat_emb, title_attn, abs_attn, title_ent_attn, abs_ent_attn], dim=1)  # (B,6,D)
        attend = F.softmax(torch.matmul(torch.tanh(torch.matmul(concat, self.trans_weight_v)), self.trans_weight_q), dim=1)  # (B,6,1)
        news_embed = torch.bmm(concat.transpose(1, 2), attend).squeeze(-1)  # (B,D)
        return news_embed

CELL 14 — NewsEncoder (Title & Abstract đều qua BERT token embeddings)

CELL 15 — NRMS model (User encoder + click predictor)

In [31]:
class NRMS(nn.Module):
    def __init__(self, news_encoder, cfg):
        super().__init__()
        self.news_encoder = news_encoder
        self.history_encoder = TextEncoder(cfg['news_final_embed_size'], cfg['num_head_text'], cfg['his_final_attn_vector_size'])
        self.user_proj = nn.Linear(cfg['news_final_embed_size'], cfg['user_embed_size'])
        
    def encode_news_batch(self, cat, subcat,
                          title_tok, title_mask,
                          abs_tok, abs_mask,
                          title_ent, title_ent_mask,
                          abs_ent, abs_ent_mask):
        # cat: (B,N) ... -> flatten -> encode -> reshape
        B, N = cat.shape
        flat = lambda x: x.view(B*N, *x.shape[2:]) if x.dim() > 2 else x.view(B*N)
        news_vec = self.news_encoder(
            flat(cat), flat(subcat),
            flat(title_tok), flat(title_mask),
            flat(abs_tok), flat(abs_mask),
            flat(title_ent), flat(title_ent_mask),
            flat(abs_ent), flat(abs_ent_mask),
        )
        return news_vec.view(B, N, -1)

    def forward(self, batch):
        dev = next(self.parameters()).device
        for k,v in batch.items():
            batch[k] = v.to(dev, non_blocking=True)

        # history (B,H,D)
        h_vec = self.encode_news_batch(
            batch["h_cat"], batch["h_subcat"],
            batch["h_title_tok"], batch["h_title_mask"],
            batch["h_abs_tok"], batch["h_abs_mask"],
            batch["h_title_ent"], batch["h_title_ent_mask"],
            batch["h_abs_ent"], batch["h_abs_ent_mask"],
        )
        h_mask = (batch["h_cat"] != 0).float()  # PAD history -> 0
        h_selfmask = h_mask.unsqueeze(1) * h_mask.unsqueeze(2)
        user_vec = self.history_encoder(h_vec, h_selfmask, h_mask)
        user_vec = self.user_proj(user_vec)  # (B,U)

        # candidates (B,C,D) -> (B,C,U)
        c_vec = self.encode_news_batch(
            batch["c_cat"], batch["c_subcat"],
            batch["c_title_tok"], batch["c_title_mask"],
            batch["c_abs_tok"], batch["c_abs_mask"],
            batch["c_title_ent"], batch["c_title_ent_mask"],
            batch["c_abs_ent"], batch["c_abs_ent_mask"],
        )
        c_vec = self.user_proj(c_vec)

        # dot product click predictor
        scores = torch.bmm(c_vec, user_vec.unsqueeze(-1)).squeeze(-1)  # (B,C)
        return scores

CELL 16 — Train/Eval loop (tối giản)

In [32]:
model = NRMS(
    news_encoder=NewsEncoder(num_cat, num_subcat, entity_embed, CFG, bert_hidden=bert_hidden),
    cfg=CFG
).to(CFG["device"])

opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
criterion = nn.CrossEntropyLoss()

In [23]:
import os, torch

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
CKPT_PATH = os.path.join(OUTPUT_DIR, "checkpoint.pt")
BEST_PATH = os.path.join(OUTPUT_DIR, "best_model.pt")

def save_ckpt(epoch, global_step, best_dev=None):
    torch.save({
        "epoch": epoch,
        "global_step": global_step,
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "best_dev": best_dev,
        "cfg": CFG,
    }, CKPT_PATH)
    print(f"[CKPT] saved epoch={epoch}, step={global_step}")

def load_ckpt():
    if os.path.exists(CKPT_PATH):
        ckpt = torch.load(CKPT_PATH, map_location=CFG["device"])
        model.load_state_dict(ckpt["model"])
        opt.load_state_dict(ckpt["opt"])
        print(f"[CKPT] loaded epoch={ckpt['epoch']}, step={ckpt['global_step']}")
        return ckpt["epoch"], ckpt["global_step"], ckpt.get("best_dev", None)
    return 0, 0, None

In [ ]:
SAVE_EVERY = 500  # lưu mỗi 500 batch

def train_one_epoch(model, loader, epoch, global_step, best_dev=None):
    model.train()
    total_loss = 0.0

    for batch_idx, batch in enumerate(tqdm(loader, desc=f"Train ep {epoch+1}")):
        opt.zero_grad(set_to_none=True)
        scores = model(batch)
        labels = batch["label"].to(scores.device)
        loss = criterion(scores, labels)
        loss.backward()
        opt.step()

        total_loss += loss.item()
        global_step += 1

        # autosave
        if global_step % SAVE_EVERY == 0:
            save_ckpt(epoch, global_step, best_dev=best_dev)

    return total_loss / max(1, len(loader)), global_step

In [36]:
import torch
from tqdm import tqdm

@torch.no_grad()
def eval_one_epoch(model, loader):
    model.eval()
    total_loss = 0.0
    for batch in tqdm(loader, desc="Eval"):
        scores = model(batch)
        labels = batch["label"].to(scores.device)
        loss = criterion(scores, labels)
        total_loss += loss.item()
    return total_loss / max(1, len(loader))

load check point to resume training

In [ ]:
import os
import shutil

OUTPUT_DIR = "/kaggle/working/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

src = "/kaggle/working/output/checkpoint.pt"
dst = os.path.join(OUTPUT_DIR, "checkpoint.pt")

if os.path.exists(dst):
    print(f"Checkpoint already exists at: {dst}")
else:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"Copied checkpoint to: {dst}")
    else:
        print(f"Source checkpoint not found: {src}")

train

In [ ]:
start_epoch, global_step, best_dev_ckpt = load_ckpt()

PATIENCE = 2
best_dev = None
bad_epochs = 0
if best_dev_ckpt is not None:
    best_dev = best_dev_ckpt

for ep in range(start_epoch, CFG["epochs"]):
    tr, global_step = train_one_epoch(model, train_loader, ep, global_step, best_dev=best_dev)
    dv = eval_one_epoch(model, dev_loader)
    print(f"Epoch {ep+1}/{CFG['epochs']} | train_loss={tr:.4f} | dev_loss={dv:.4f}")

    improved = (best_dev is None) or (dv < best_dev - 1e-6)
    if improved:
        best_dev = dv
        bad_epochs = 0
        torch.save(model.state_dict(), BEST_PATH)
        print("[BEST] saved best_model.pt")
    else:
        bad_epochs += 1
        print(f"[EARLY STOP] no improve: {bad_epochs}/{PATIENCE}")

    save_ckpt(ep+1, global_step, best_dev=best_dev)

    if bad_epochs >= PATIENCE:
        print(f"Early stopping at epoch {ep+1}. Best dev_loss={best_dev:.4f}")
        break


In [ ]:
print(os.listdir("/kaggle/working/output"))

best model

In [33]:
BEST_PATH = "/kaggle/input/models/giahnhong/best-model-l/tensorflow2/default/1/best_model (3).pt" 
model.load_state_dict(torch.load(BEST_PATH, map_location=CFG["device"]))
model.to(CFG["device"])
model.eval()
print("Loaded best_model.pt")

Loaded best_model.pt


metrics

In [34]:
from sklearn.metrics import roc_auc_score

def dcg_score(y_true, y_score, k=10):
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order[:k])
    gains = 2 ** y_true - 1
    discounts = np.log2(np.arange(len(y_true)) + 2)
    return np.sum(gains / discounts)

def ndcg_score(y_true, y_score, k=10):
    best = dcg_score(y_true, y_true, k)
    actual = dcg_score(y_true, y_score, k)
    return actual / best if best > 0 else 0.0

def mrr_score(y_true, y_score):
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order)
    for i, val in enumerate(y_true):
        if val == 1:
            return 1.0 / (i + 1)
    return 0.0

print('Metric functions defined')

Metric functions defined


In [53]:
from tqdm import tqdm
import pandas as pd

def prepare_test_data(behaviors):
    test_samples = []
    for _, row in tqdm(behaviors.iterrows(), total=len(behaviors), desc='Preparing test data'):
        history = str(row['History']).split() if pd.notna(row['History']) else []
        impressions = str(row['Impressions']).split() if pd.notna(row['Impressions']) else []
        if len(impressions) == 0:
            continue

        candidates, labels = [], []
        for imp in impressions:
            parts = imp.split('-')
            if len(parts) == 2:
                candidates.append(parts[0])
                labels.append(int(parts[1]))

        # chỉ lấy sample có ít nhất 1 positive
        if len(candidates) > 0 and sum(labels) > 0:
            test_samples.append({
                'history': history[-MAX_HIS:],     # dùng MAX_HIS của notebook
                'candidates': candidates,
                'labels': labels
            })
    return test_samples

test_data = prepare_test_data(beh_dev)   # behaviors_dev trong notebook thuần-BERT đang là beh_dev
print(f'Test samples: {len(test_data)}')

Preparing test data: 100%|██████████| 376471/376471 [00:26<00:00, 13948.22it/s]


Test samples: 376471


In [54]:
import torch

@torch.no_grad()
def build_eval_batch(sample, news_feat, device):
    history = sample['history'] if sample['history'] else ['PAD']
    candidates = sample['candidates']
    labels = sample['labels']

    # pad history đúng MAX_HIS
    if len(history) >= MAX_HIS:
        history = history[-MAX_HIS:]
    else:
        history = ["PAD"]*(MAX_HIS-len(history)) + history

    h = [news_feat.get(nid, news_feat["PAD"]) for nid in history]
    c = [news_feat.get(nid, news_feat["PAD"]) for nid in candidates]

    def stack(feats, key):
        return torch.stack([f[key] for f in feats], dim=0)

    batch = {
        # history: (1,H,...)
        "h_cat": torch.tensor([[f["cat"] for f in h]], dtype=torch.long),
        "h_subcat": torch.tensor([[f["subcat"] for f in h]], dtype=torch.long),
        "h_title_tok": stack(h, "title_tok").unsqueeze(0),
        "h_title_mask": stack(h, "title_mask").unsqueeze(0),
        "h_abs_tok": stack(h, "abs_tok").unsqueeze(0),
        "h_abs_mask": stack(h, "abs_mask").unsqueeze(0),
        "h_title_ent": stack(h, "title_ent").unsqueeze(0),
        "h_title_ent_mask": stack(h, "title_ent_mask").unsqueeze(0),
        "h_abs_ent": stack(h, "abs_ent").unsqueeze(0),
        "h_abs_ent_mask": stack(h, "abs_ent_mask").unsqueeze(0),

        # candidates: (1,C,...)
        "c_cat": torch.tensor([[f["cat"] for f in c]], dtype=torch.long),
        "c_subcat": torch.tensor([[f["subcat"] for f in c]], dtype=torch.long),
        "c_title_tok": stack(c, "title_tok").unsqueeze(0),
        "c_title_mask": stack(c, "title_mask").unsqueeze(0),
        "c_abs_tok": stack(c, "abs_tok").unsqueeze(0),
        "c_abs_mask": stack(c, "abs_mask").unsqueeze(0),
        "c_title_ent": stack(c, "title_ent").unsqueeze(0),
        "c_title_ent_mask": stack(c, "title_ent_mask").unsqueeze(0),
        "c_abs_ent": stack(c, "abs_ent").unsqueeze(0),
        "c_abs_ent_mask": stack(c, "abs_ent_mask").unsqueeze(0),

        # label không dùng ở eval từng sample, nhưng giữ format
        "label": torch.tensor([0], dtype=torch.long),
    }

    # move to device
    for k,v in batch.items():
        batch[k] = v.to(device, non_blocking=True)
    return batch, np.array(labels, dtype=np.int64)


def test_model(model, test_data, news_feat, device):
    model.eval()
    all_auc, all_mrr, all_ndcg5, all_ndcg10 = [], [], [], []

    with torch.no_grad():
        for sample in tqdm(test_data, desc='Testing'):
            batch, labels = build_eval_batch(sample, news_feat, device)

            scores = model(batch).squeeze(0).detach().cpu().numpy()  # (C,)

            try:
                if len(np.unique(labels)) > 1:
                    all_auc.append(roc_auc_score(labels, scores))
            except:
                pass

            all_mrr.append(mrr_score(labels, scores))
            all_ndcg5.append(ndcg_score(labels, scores, k=5))
            all_ndcg10.append(ndcg_score(labels, scores, k=10))

    return {
        'AUC': np.mean(all_auc) if all_auc else 0,
        'MRR': np.mean(all_mrr),
        'nDCG@5': np.mean(all_ndcg5),
        'nDCG@10': np.mean(all_ndcg10)
    }

In [55]:
OUTPUT_DIR = "./output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

best_path = os.path.join(OUTPUT_DIR, 'best_model.pt')
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=CFG["device"]))
    print('Loaded best model')

results = test_model(model, test_data, news_feat, CFG["device"])

print('\n' + '='*50)
print('TEST RESULTS')
print('='*50)
for m, v in results.items():
    print(f'{m}: {v:.4f}')

Testing: 100%|██████████| 376471/376471 [1:47:05<00:00, 58.59it/s]  



TEST RESULTS
AUC: 0.6847
MRR: 0.3841
nDCG@5: 0.3672
nDCG@10: 0.4312


In [57]:
# Compare with paper results
paper = {'AUC': 0.6785, 'MRR': 0.3340, 'nDCG@5': 0.3676, 'nDCG@10': 0.4300}

print('\n' + '='*60)
print('COMPARISON WITH NRMS PAPER (EMNLP 2019)')
print('='*60)
print(f'{"Metric":<12}{"Ours":<12}{"Paper":<12}{"Diff":<12}')
print('-'*60)
for m in paper:
    diff = results[m] - paper[m]
    print(f'{m:<12}{results[m]:<12.4f}{paper[m]:<12.4f}{diff:+.4f}')
print('='*60)


COMPARISON WITH NRMS PAPER (EMNLP 2019)
Metric      Ours        Paper       Diff        
------------------------------------------------------------
AUC         0.6847      0.6785      +0.0062
MRR         0.3841      0.3340      +0.0501
nDCG@5      0.3672      0.3676      -0.0004
nDCG@10     0.4312      0.4300      +0.0012


In [ ]:
import matplotlib.pyplot as plt

metrics = list(paper.keys())
ours = [results[m] for m in metrics]
theirs = [paper[m] for m in metrics]

x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - 0.175, ours, 0.35, label='Our Results', color='steelblue')
b2 = ax.bar(x + 0.175, theirs, 0.35, label='Paper Results', color='coral')

ax.set_ylabel('Score')
ax.set_title('NRMS: Our Results vs Paper (EMNLP 2019)')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(b1, ours):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=9)
for bar, val in zip(b2, theirs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'metrics_comparison.png'), dpi=150)
plt.show()

In [ ]:
# Compare with baseline results
baseline = {'AUC': 0.6174, 'MRR': 0.3395, 'nDCG@5': 0.3216, 'nDCG@10': 0.3829}

print('\n' + '='*60)
print('COMPARISON WITH BASELINE')
print('='*60)
print(f'{"Metric":<12}{"Enhanced":<12}{"Baseline":<12}{"Diff":<12}')
print('-'*60)
for m in baseline:
    diff = results[m] - baseline[m]
    print(f'{m:<12}{results[m]:<12.4f}{baseline[m]:<12.4f}{diff:+.4f}')
print('='*60)

In [ ]:
import matplotlib.pyplot as plt

metrics = list(baseline.keys())
ours = [results[m] for m in metrics]
theirs = [baseline[m] for m in metrics]

x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - 0.175, ours, 0.35, label='Enhanced Results', color='steelblue')
b2 = ax.bar(x + 0.175, theirs, 0.35, label='Baseline', color='coral')

ax.set_ylabel('Score')
ax.set_title('NRMS: Our Enhanced Results vs Baseline')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(b1, ours):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=9)
for bar, val in zip(b2, theirs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{val:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'metrics_comparison.png'), dpi=150)
plt.show()

In [ ]:
# Compare with baseline results
baseline = {'AUC': 0.6618, 'MRR': 0.3695, 'nDCG@5': 0.3503, 'nDCG@10': 0.4138}

print('\n' + '='*60)
print('COMPARISON WITH freeze bert')
print('='*60)
print(f'{"Metric":<12}{"fine tuned":<12}{"freeze bert":<12}{"Diff":<12}')
print('-'*60)
for m in baseline:
    diff = results[m] - baseline[m]
    print(f'{m:<12}{results[m]:<12.4f}{baseline[m]:<12.4f}{diff:+.4f}')
print('='*60)